In [ ]:
import os
import sys
import shutil
import zipfile
import importlib
from pathlib import Path

from google.colab import files
from google.colab import drive

PILOT_ROOT = "/content/pilot_full"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)

with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# Upload the active config.py.
print("Upload config.py")
uploaded_config = files.upload()

config_upload_name = next(
    (name for name in uploaded_config.keys()
     if name.startswith("config") and name.endswith(".py")),
    None,
)

if config_upload_name is None:
    raise FileNotFoundError(
        f"No config*.py file was uploaded. Uploaded files: {list(uploaded_config.keys())}"
    )

config_target_path = os.path.join(PILOT_ROOT, "config.py")
shutil.copy(os.path.join("/content", config_upload_name), config_target_path)

import pilot_full.config as cfg
importlib.reload(cfg)

drive.mount("/content/drive")

STEP6_OUTPUT_DIR = cfg.STEP6_OUTPUT_DIR
STEP6_INPUT_DIR = cfg.STEP6_INPUT_DIR
STEP6_SAMPLE_FAMILY = cfg.STEP6_SAMPLE_FAMILY

STEP6_USE_STEP5_ZIP_INPUT = bool(getattr(cfg, "STEP6_USE_STEP5_ZIP_INPUT", False))
STEP6_INPUT_ZIP_PATHS = getattr(cfg, "STEP6_INPUT_ZIP_PATHS", {})

if os.path.exists(STEP6_OUTPUT_DIR):
    shutil.rmtree(STEP6_OUTPUT_DIR)
os.makedirs(STEP6_OUTPUT_DIR, exist_ok=True)

# Use the sample family specified by the current config.
# For this run, it is expected to be "pair".
os.makedirs(STEP6_INPUT_DIR, exist_ok=True)

if STEP6_USE_STEP5_ZIP_INPUT:
    if STEP6_SAMPLE_FAMILY not in STEP6_INPUT_ZIP_PATHS:
        raise KeyError(
            f"Missing zip path for active sample family {STEP6_SAMPLE_FAMILY!r} "
            "in cfg.STEP6_INPUT_ZIP_PATHS."
        )

    zip_path = STEP6_INPUT_ZIP_PATHS[STEP6_SAMPLE_FAMILY]

    if not os.path.exists(zip_path):
        raise FileNotFoundError(
            f"Step5 zip not found for {STEP6_SAMPLE_FAMILY!r}: {zip_path}"
        )

    for old_path in Path(STEP6_INPUT_DIR).glob("*"):
        if old_path.is_file():
            old_path.unlink()
        elif old_path.is_dir():
            shutil.rmtree(old_path)

    print("\nExtracting Step5 zip")
    print("sample_family:", STEP6_SAMPLE_FAMILY)
    print("zip_path:", zip_path)
    print("target_dir:", STEP6_INPUT_DIR)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(STEP6_INPUT_DIR)

pt_files = sorted(Path(STEP6_INPUT_DIR).rglob("*.pt"))

if not pt_files:
    raise FileNotFoundError(
        f"No Step5 .pt files found in STEP6_INPUT_DIR:\n{STEP6_INPUT_DIR}"
    )

print("\nSetup complete.")
print("Uploaded config:", config_upload_name)
print("Config copied to:", config_target_path)
print("STEP6_SAMPLE_FAMILY:", STEP6_SAMPLE_FAMILY)
print("STEP6_INPUT_DIR:", STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", STEP6_OUTPUT_DIR)
print("STEP6_USE_STEP5_ZIP_INPUT:", STEP6_USE_STEP5_ZIP_INPUT)
print("Discovered .pt files:", len(pt_files))

for p in pt_files[:10]:
    print("-", p.relative_to(STEP6_INPUT_DIR))
if len(pt_files) > 10:
    print("...")

Upload config.py


Saving config_positional_context_cross_scs_full_drive.py to config_positional_context_cross_scs_full_drive.py
Mounted at /content/drive

Setup complete.
Uploaded config: config_positional_context_cross_scs_full_drive.py
Config copied to: /content/pilot_full/config.py
STEP6_SAMPLE_FAMILY: pair
STEP6_INPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_3b
STEP6_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only
STEP6_USE_STEP5_ZIP_INPUT: False
Discovered .pt files: 118
- FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
- FloorPlan11_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
- FloorPlan12_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
- FloorPlan13_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
- FloorPlan14_step4_probe_sa

In [ ]:
import os
import sys
import gc
import json
import time
import random
import shutil
import importlib
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_full.config as cfg
importlib.reload(cfg)

# Experiment identity
EXPERIMENT_NAME = cfg.STEP6_EXPERIMENT_NAME
EXPERIMENT_ID = getattr(cfg, "STEP6_EXPERIMENT_ID", EXPERIMENT_NAME)
EXPERIMENT_SHORT_NAME = getattr(cfg, "STEP6_EXPERIMENT_SHORT_NAME", EXPERIMENT_NAME)
EXPERIMENT_DESCRIPTION = cfg.STEP6_EXPERIMENT_DESCRIPTION
SCRIPT_FAMILY = getattr(cfg, "STEP6_SCRIPT_FAMILY", "pair_relation_layerwise_logreg")

SUPPORTED_SCRIPT_FAMILIES = {"pair_relation_layerwise_logreg"}
if SCRIPT_FAMILY not in SUPPORTED_SCRIPT_FAMILIES:
    raise ValueError(
        f"Unsupported STEP6_SCRIPT_FAMILY={SCRIPT_FAMILY!r}. "
        f"Supported: {sorted(SUPPORTED_SCRIPT_FAMILIES)}"
    )

RANDOM_SEED = cfg.RANDOM_SEED
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

MODEL_NAME = cfg.STEP6_MODEL_NAME
MODEL_TAG = cfg.STEP6_MODEL_TAG

SAMPLE_FAMILY = cfg.STEP6_SAMPLE_FAMILY
SAMPLE_FAMILIES = list(getattr(cfg, "STEP6_SAMPLE_FAMILIES", [SAMPLE_FAMILY]))

FEATURE_KEY = cfg.STEP6_FEATURE_KEY
LABEL_FIELD = cfg.STEP6_LABEL_FIELD

EXPLICIT_DIRECT = cfg.EXPLICIT_DIRECT
KEEP_EVIDENCE_TYPES = set(cfg.STEP6_KEEP_EVIDENCE_TYPES)

DROP_NONE_LABEL = bool(cfg.STEP6_DROP_NONE_LABEL)
DROP_EMPTY_LABEL = bool(cfg.STEP6_DROP_EMPTY_LABEL)
NONE_RELATION_LABEL = getattr(cfg, "NONE_RELATION_LABEL", "none")

ALLOWED_LABELS_RAW = getattr(cfg, "STEP6_ALLOWED_LABELS", None)
ALLOWED_LABELS = None if ALLOWED_LABELS_RAW is None else set(ALLOWED_LABELS_RAW)

TRAIN_SCENES = list(cfg.STEP6_TRAIN_SCENES)
TEST_SCENES = list(cfg.STEP6_TEST_SCENES)
REQUIRE_EXPLICIT_SCENE_SPLIT = bool(cfg.STEP6_REQUIRE_EXPLICIT_SCENE_SPLIT)


TEST_ABLATION_MODE = getattr(
    cfg,
    "STEP6_TEST_ABLATION_MODE",
    "none",
)

TEST_PAIR_COVERAGE_KEY = getattr(
    cfg,
    "STEP6_TEST_PAIR_COVERAGE_KEY",
    "directed_type_pair",
)

TEST_INCLUDE_FULL_BATHROOM = bool(
    getattr(cfg, "STEP6_TEST_INCLUDE_FULL_BATHROOM", True)
)

TEST_SUBSET_OVERALL_NAME = getattr(
    cfg,
    "STEP6_TEST_SUBSET_OVERALL_NAME",
    "overall",
)

TEST_SUBSET_SEEN_NAME = getattr(
    cfg,
    "STEP6_TEST_SUBSET_SEEN_NAME",
    "seen_train_directed_type_pair",
)

TEST_SUBSET_UNSEEN_NAME = getattr(
    cfg,
    "STEP6_TEST_SUBSET_UNSEEN_NAME",
    "unseen_train_directed_type_pair",
)

TEST_SUBSETS_TO_EVALUATE = list(
    getattr(
        cfg,
        "STEP6_TEST_SUBSETS_TO_EVALUATE",
        [TEST_SUBSET_OVERALL_NAME],
    )
)

SUPPORTED_TEST_SUBSETS = {
    TEST_SUBSET_OVERALL_NAME,
    TEST_SUBSET_SEEN_NAME,
    TEST_SUBSET_UNSEEN_NAME,
}

MAX_ITER = cfg.STEP6_LOGREG_MAX_ITER
C_VALUE = cfg.STEP6_LOGREG_C
CLASS_WEIGHT = getattr(cfg, "STEP6_LOGREG_CLASS_WEIGHT", "balanced")
SOLVER = getattr(cfg, "STEP6_LOGREG_SOLVER", "lbfgs")

PRINT_DATASET_SUMMARY = bool(cfg.STEP6_PRINT_DATASET_SUMMARY)
PRINT_LAYER_PROGRESS = bool(cfg.STEP6_PRINT_LAYER_PROGRESS)
PRINT_TOP_LAYERS = bool(cfg.STEP6_PRINT_TOP_LAYERS)
NUM_TOP_LAYERS_TO_PRINT = int(cfg.STEP6_NUM_TOP_LAYERS_TO_PRINT)

USE_LOCAL_CONTENT_CACHE = bool(getattr(cfg, "STEP6_USE_LOCAL_CONTENT_CACHE", False))
LOCAL_CONTENT_CACHE_ROOT = getattr(
    cfg,
    "STEP6_LOCAL_CONTENT_CACHE_ROOT",
    f"/content/step5_hs_cache_{MODEL_TAG}_{EXPERIMENT_SHORT_NAME}",
)
LOCAL_CACHE_SAFETY_MARGIN_GB = float(
    getattr(cfg, "STEP6_LOCAL_CACHE_SAFETY_MARGIN_GB", 10.0)
)

LAYER_INDICES_TO_RUN = getattr(cfg, "STEP6_LAYER_INDICES_TO_RUN", None)

STEP6_INPUT_DIR = cfg.STEP6_INPUT_DIR
STEP6_OUTPUT_DIR = cfg.STEP6_OUTPUT_DIR

STEP6_RESULT_JSON = cfg.STEP6_RESULT_JSON
STEP6_LAYER_SCORES_CSV = cfg.STEP6_LAYER_SCORES_CSV
STEP6_LABEL_METRICS_CSV = cfg.STEP6_LABEL_METRICS_CSV
STEP6_RECALL_MATRIX_LONG_CSV = cfg.STEP6_RECALL_MATRIX_LONG_CSV
STEP6_TEST_PREDICTIONS_CSV = cfg.STEP6_TEST_PREDICTIONS_CSV
STEP6_MANIFEST_JSON = cfg.STEP6_MANIFEST_JSON


def validate_step6_config() -> None:
    if SAMPLE_FAMILY != "pair":
        raise ValueError(f"This notebook expects STEP6_SAMPLE_FAMILY='pair', got {SAMPLE_FAMILY!r}.")

    if SAMPLE_FAMILIES != ["pair"]:
        raise ValueError(f"This notebook expects STEP6_SAMPLE_FAMILIES=['pair'], got {SAMPLE_FAMILIES!r}.")

    if FEATURE_KEY != "layer_diff_features":
        raise ValueError(f"This notebook expects STEP6_FEATURE_KEY='layer_diff_features', got {FEATURE_KEY!r}.")

    if LABEL_FIELD != "relation":
        raise ValueError(f"This notebook expects STEP6_LABEL_FIELD='relation', got {LABEL_FIELD!r}.")

    expected_evidence = {EXPLICIT_DIRECT}
    if KEEP_EVIDENCE_TYPES != expected_evidence:
        raise ValueError(
            "Current direct-only probe expects STEP6_KEEP_EVIDENCE_TYPES "
            f"to be {sorted(expected_evidence)}, got {sorted(KEEP_EVIDENCE_TYPES)}."
        )

    if not DROP_NONE_LABEL:
        raise ValueError("Current non-none relation probe expects STEP6_DROP_NONE_LABEL=True.")

    if not DROP_EMPTY_LABEL:
        raise ValueError("Current non-empty relation probe expects STEP6_DROP_EMPTY_LABEL=True.")

    if REQUIRE_EXPLICIT_SCENE_SPLIT and (not TRAIN_SCENES or not TEST_SCENES):
        raise ValueError("STEP6_TRAIN_SCENES and STEP6_TEST_SCENES must be defined.")


    unsupported_test_subsets = sorted(
        set(TEST_SUBSETS_TO_EVALUATE) - SUPPORTED_TEST_SUBSETS
        )

    if unsupported_test_subsets:
      raise ValueError(
          "Unsupported STEP6_TEST_SUBSETS_TO_EVALUATE values: "
          f"{unsupported_test_subsets}. "
          f"Supported: {sorted(SUPPORTED_TEST_SUBSETS)}"
          )

    if TEST_ABLATION_MODE == "none":
      if TEST_SUBSETS_TO_EVALUATE != [TEST_SUBSET_OVERALL_NAME]:
        raise ValueError(
            "When STEP6_TEST_ABLATION_MODE='none', "
            "STEP6_TEST_SUBSETS_TO_EVALUATE must be ['overall']."
            )

    if TEST_ABLATION_MODE == "seen_unseen_train_directed_type_pair":
      required_subset_names = {
          TEST_SUBSET_SEEN_NAME,
          TEST_SUBSET_UNSEEN_NAME,
          }

      if not (set(TEST_SUBSETS_TO_EVALUATE) & required_subset_names):
        raise ValueError(
            "seen_unseen_train_directed_type_pair mode should evaluate "
            "at least one of seen or unseen subsets."
            )

    overlap = set(TRAIN_SCENES) & set(TEST_SCENES)
    if overlap:
        raise ValueError(f"STEP6_TRAIN_SCENES and STEP6_TEST_SCENES overlap: {sorted(overlap)}")

    if TEST_ABLATION_MODE not in {"none", "seen_unseen_train_directed_type_pair"}:
        raise ValueError(
            "Unsupported STEP6_TEST_ABLATION_MODE: "
            f"{TEST_ABLATION_MODE!r}"
        )

    if TEST_ABLATION_MODE == "seen_unseen_train_directed_type_pair":
        if TEST_PAIR_COVERAGE_KEY != "directed_type_pair":
            raise ValueError(
                "seen/unseen ablation currently expects "
                "STEP6_TEST_PAIR_COVERAGE_KEY='directed_type_pair'. "
                f"Got: {TEST_PAIR_COVERAGE_KEY!r}"
            )

    if not os.path.isdir(STEP6_INPUT_DIR):
        raise FileNotFoundError(f"STEP6_INPUT_DIR does not exist: {STEP6_INPUT_DIR}")

    os.makedirs(STEP6_OUTPUT_DIR, exist_ok=True)


validate_step6_config()

print("Step6 config validated.")
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("EXPERIMENT_ID:", EXPERIMENT_ID)
print("EXPERIMENT_SHORT_NAME:", EXPERIMENT_SHORT_NAME)
print("SCRIPT_FAMILY:", SCRIPT_FAMILY)
print("MODEL:", MODEL_NAME, "|", MODEL_TAG)
print("SAMPLE_FAMILY:", SAMPLE_FAMILY)
print("FEATURE_KEY:", FEATURE_KEY)
print("LABEL_FIELD:", LABEL_FIELD)
print("KEEP_EVIDENCE_TYPES:", sorted(KEEP_EVIDENCE_TYPES))
print("ALLOWED_LABELS:", None if ALLOWED_LABELS is None else sorted(ALLOWED_LABELS))
print("TRAIN_SCENES:", len(TRAIN_SCENES))
print("TEST_SCENES:", len(TEST_SCENES))
print("TEST_ABLATION_MODE:", TEST_ABLATION_MODE)
print("TEST_PAIR_COVERAGE_KEY:", TEST_PAIR_COVERAGE_KEY)
print("TEST_INCLUDE_FULL_BATHROOM:", TEST_INCLUDE_FULL_BATHROOM)
print("TEST_SUBSET_OVERALL_NAME:", TEST_SUBSET_OVERALL_NAME)
print("TEST_SUBSET_SEEN_NAME:", TEST_SUBSET_SEEN_NAME)
print("TEST_SUBSET_UNSEEN_NAME:", TEST_SUBSET_UNSEEN_NAME)
print("STEP6_INPUT_DIR:", STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", STEP6_OUTPUT_DIR)
print("USE_LOCAL_CONTENT_CACHE:", USE_LOCAL_CONTENT_CACHE)
print("LAYER_INDICES_TO_RUN:", LAYER_INDICES_TO_RUN)
print("TEST_SUBSETS_TO_EVALUATE:", TEST_SUBSETS_TO_EVALUATE)

Step6 config validated.
EXPERIMENT_NAME: step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only
EXPERIMENT_ID: pair_explicit_direct_relation_ld_test_seen_unseen_dtp_only
EXPERIMENT_SHORT_NAME: pair_ed_rel_ld_seen_unseen_dtp_only
SCRIPT_FAMILY: pair_relation_layerwise_logreg
MODEL: Qwen/Qwen2.5-3B | qwen2_5_3b
SAMPLE_FAMILY: pair
FEATURE_KEY: layer_diff_features
LABEL_FIELD: relation
KEEP_EVIDENCE_TYPES: ['explicit_direct']
ALLOWED_LABELS: ['above', 'below', 'contains', 'in', 'left_of', 'on', 'right_of', 'supports']
TRAIN_SCENES: 90
TEST_SCENES: 30
TEST_ABLATION_MODE: seen_unseen_train_directed_type_pair
TEST_PAIR_COVERAGE_KEY: directed_type_pair
TEST_INCLUDE_FULL_BATHROOM: False
TEST_SUBSET_OVERALL_NAME: overall
TEST_SUBSET_SEEN_NAME: seen_train_directed_type_pair
TEST_SUBSET_UNSEEN_NAME: unseen_train_directed_type_pair
STEP6_INPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/st

In [ ]:
# Utilities for JSON handling, file discovery, caching, and filtering.

def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, set):
        return sorted(make_json_safe(v) for v in obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def summarize_counter(values) -> Dict[str, int]:
    return {str(k): int(v) for k, v in Counter(values).most_common()}


def discover_step5_pt_files() -> List[Dict[str, Any]]:
    pt_paths = sorted(Path(STEP6_INPUT_DIR).rglob("*.pt"))

    if not pt_paths:
        raise FileNotFoundError(f"No .pt files found in STEP6_INPUT_DIR: {STEP6_INPUT_DIR}")

    return [
        {
            "sample_family": SAMPLE_FAMILY,
            "input_dir": STEP6_INPUT_DIR,
            "path": str(path),
            "drive_path": str(path),
            "relative_path": str(path.relative_to(STEP6_INPUT_DIR)),
        }
        for path in pt_paths
    ]


def remount_google_drive_if_needed():
    from google.colab import drive
    print("Attempting Google Drive remount after I/O error...")
    drive.mount("/content/drive", force_remount=True)


def copy_file_with_retry(
    src_path: str,
    dst_path: str,
    expected_size: int,
    max_retries: int = 5,
    sleep_seconds: int = 10,
) -> str:
    if os.path.exists(dst_path) and os.path.getsize(dst_path) == expected_size:
        return "reused"

    os.makedirs(os.path.dirname(dst_path), exist_ok=True)

    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            tmp_path = dst_path + ".tmp"
            if os.path.exists(tmp_path):
                os.remove(tmp_path)

            shutil.copy2(src_path, tmp_path)

            actual_size = os.path.getsize(tmp_path)
            if actual_size != expected_size:
                raise IOError(
                    f"Size mismatch after copy: expected={expected_size}, actual={actual_size}"
                )

            os.replace(tmp_path, dst_path)
            return "copied"

        except Exception as e:
            last_error = e
            print(f"Copy attempt {attempt}/{max_retries} failed: {repr(e)}", flush=True)

            if getattr(e, "errno", None) == 107:
                remount_google_drive_if_needed()

            if attempt < max_retries:
                time.sleep(sleep_seconds)

    raise RuntimeError(
        f"Failed to copy after {max_retries} attempts:\n"
        f"src={src_path}\n"
        f"dst={dst_path}\n"
        f"last_error={repr(last_error)}"
    )


def maybe_copy_step5_files_to_local_cache(
    drive_pt_file_records: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    if not USE_LOCAL_CONTENT_CACHE:
        print("USE_LOCAL_CONTENT_CACHE=False. Reading .pt files from Drive.")
        return drive_pt_file_records

    total_size_bytes = sum(os.path.getsize(item["path"]) for item in drive_pt_file_records)
    total_size_gb = total_size_bytes / 1024**3
    free_gb = shutil.disk_usage("/content").free / 1024**3

    print("Local cache pre-check")
    print("Total Step5 .pt size GB:", round(total_size_gb, 3))
    print("Free /content disk GB:", round(free_gb, 3))
    print("Safety margin GB:", LOCAL_CACHE_SAFETY_MARGIN_GB)
    print("Cache root:", LOCAL_CONTENT_CACHE_ROOT)

    if total_size_gb + LOCAL_CACHE_SAFETY_MARGIN_GB > free_gb:
        raise RuntimeError(
            "Not enough /content disk space for local cache. "
            "Set STEP6_USE_LOCAL_CONTENT_CACHE=False or run a smaller layer/file subset."
        )

    os.makedirs(LOCAL_CONTENT_CACHE_ROOT, exist_ok=True)

    cached_records = []
    copied_count = 0
    reused_count = 0

    for i, item in enumerate(drive_pt_file_records, start=1):
        src_path = item["path"]
        relative_path = item["relative_path"]
        expected_size = os.path.getsize(src_path)

        dst_dir = os.path.join(LOCAL_CONTENT_CACHE_ROOT, SAMPLE_FAMILY)
        dst_path = os.path.join(dst_dir, os.path.basename(relative_path))

        print(f"[{i}/{len(drive_pt_file_records)}] cache {relative_path}", flush=True)

        status = copy_file_with_retry(
            src_path=src_path,
            dst_path=dst_path,
            expected_size=expected_size,
        )

        copied_count += int(status == "copied")
        reused_count += int(status == "reused")

        cached_item = dict(item)
        cached_item["drive_path"] = src_path
        cached_item["path"] = dst_path
        cached_item["input_dir"] = dst_dir
        cached_item["relative_path"] = os.path.basename(relative_path)

        cached_records.append(cached_item)

    print("Local cache ready.")
    print("Copied:", copied_count)
    print("Reused:", reused_count)

    return cached_records


def keep_record_for_step6(rec: Dict[str, Any]) -> bool:
    evidence_type = rec.get("evidence_type")

    if evidence_type not in KEEP_EVIDENCE_TYPES:
        return False

    label = rec.get(LABEL_FIELD)

    if DROP_EMPTY_LABEL and (label is None or str(label).strip() == ""):
        return False

    label = str(label)

    if DROP_NONE_LABEL and label == NONE_RELATION_LABEL:
        return False

    if ALLOWED_LABELS is not None and label not in ALLOWED_LABELS:
        return False

    return True

In [ ]:
FEATURE_KEYS_TO_DROP = {
    "layer_subject_features",
    "layer_object_features",
    "layer_diff_features",
    "layer_concat_features",
}

drive_pt_file_records = discover_step5_pt_files()
pt_file_records = maybe_copy_step5_files_to_local_cache(drive_pt_file_records)

print("Active Step5 .pt files:", len(pt_file_records))
for item in pt_file_records[:5]:
    print("-", item["path"])
if len(pt_file_records) > 5:
    print("...")

all_records_meta = []
source_file_summaries = []

NUM_LAYERS = None
FEATURE_DIM = None

raw_record_count_total = 0
kept_record_count_total = 0

print("\nScanning Step5 .pt payloads metadata only...")
print("=" * 100)

for file_i, item in enumerate(pt_file_records, start=1):
    pt_path = item["path"]
    drive_path = item.get("drive_path", pt_path)
    input_dir = item["input_dir"]
    relative_path = item["relative_path"]

    file_size_mb = os.path.getsize(pt_path) / 1024**2

    print(
        f"\n[{file_i}/{len(pt_file_records)}] {relative_path} | "
        f"size={file_size_mb:.2f} MB",
        flush=True,
    )

    try:
        payload = torch.load(pt_path, map_location="cpu")
    except Exception as e:
        raise RuntimeError(
            f"Failed to load Step5 .pt file:\n"
            f"active_path={pt_path}\n"
            f"drive_path={drive_path}\n"
            f"error={repr(e)}"
        ) from e

    if "records" not in payload:
        raise KeyError(
            f"Expected key 'records' in Step5 payload. "
            f"file={relative_path}, payload_keys={sorted(payload.keys())}"
        )

    records = payload["records"]
    raw_record_count_total += len(records)

    kept_in_file = 0

    for local_record_idx, rec in enumerate(records):
        if not keep_record_for_step6(rec):
            continue

        if FEATURE_KEY not in rec:
            raise KeyError(
                f"Feature key {FEATURE_KEY!r} missing in kept record. "
                f"file={relative_path}, local_record_idx={local_record_idx}"
            )

        feature_arr = np.asarray(rec[FEATURE_KEY])

        if feature_arr.ndim != 2:
            raise ValueError(
                f"{FEATURE_KEY} must be 2D [num_layers, feature_dim], "
                f"got shape={feature_arr.shape} in {relative_path}"
            )

        if NUM_LAYERS is None:
            NUM_LAYERS = int(feature_arr.shape[0])
            FEATURE_DIM = int(feature_arr.shape[1])
            print(f"Detected feature shape: NUM_LAYERS={NUM_LAYERS}, FEATURE_DIM={FEATURE_DIM}")

        elif feature_arr.shape != (NUM_LAYERS, FEATURE_DIM):
            raise ValueError(
                f"Inconsistent feature shape in {relative_path}: "
                f"expected {(NUM_LAYERS, FEATURE_DIM)}, got {feature_arr.shape}"
            )

        meta = {}
        for k, v in rec.items():
            if k in FEATURE_KEYS_TO_DROP:
                continue
            if isinstance(v, (torch.Tensor, np.ndarray)):
                continue
            meta[k] = v

        meta["_step6_sample_family"] = SAMPLE_FAMILY
        meta["_source_step5_pt_file"] = relative_path
        meta["_source_step5_input_dir"] = input_dir
        meta["_source_step5_absolute_pt_file"] = pt_path
        meta["_source_step5_drive_pt_file"] = drive_path
        meta["_source_local_record_idx"] = int(local_record_idx)

        all_records_meta.append(meta)
        kept_in_file += 1

    kept_record_count_total += kept_in_file

    source_file_summaries.append({
        "sample_family": SAMPLE_FAMILY,
        "pt_file": relative_path,
        "absolute_pt_file": pt_path,
        "drive_pt_file": drive_path,
        "input_dir": input_dir,
        "source_file": payload.get("source_file"),
        "scene": payload.get("scene"),
        "source_type": payload.get("source_type"),
        "model_name": payload.get("model_name"),
        "model_tag": payload.get("model_tag"),
        "num_records_raw": len(records),
        "num_records_kept_after_step6_filter": kept_in_file,
        "payload_keys": sorted(payload.keys()),
        "file_size_mb": file_size_mb,
    })

    print(
        f"raw_records={len(records)} | kept_after_filter={kept_in_file} | "
        f"total_kept={len(all_records_meta)}",
        flush=True,
    )

    del payload
    del records
    gc.collect()

if NUM_LAYERS is None or FEATURE_DIM is None or not all_records_meta:
    raise ValueError("No valid records found after Step6 filtering.")

metadata = all_records_meta
y = np.asarray([str(meta[LABEL_FIELD]) for meta in metadata])
label_order = sorted(set(y.tolist()))

print("\nFinished metadata scan.")
print("Raw records before filtering:", raw_record_count_total)
print("Filtered records for Step6:", len(metadata))
print("Dropped records:", raw_record_count_total - len(metadata))
print("NUM_LAYERS:", NUM_LAYERS)
print("FEATURE_DIM:", FEATURE_DIM)
print("Label order:", label_order)

Local cache pre-check
Total Step5 .pt size GB: 108.574
Free /content disk GB: 205.494
Safety margin GB: 20.0
Cache root: /content/step5_hs_cache_qwen2_5_3b_pair_ed_rel_ld_seen_unseen_dtp_only
[1/118] cache FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
[2/118] cache FloorPlan11_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
Copy attempt 1/5 failed: OSError(107, 'Transport endpoint is not connected')
Attempting Google Drive remount after I/O error...
Mounted at /content/drive
[3/118] cache FloorPlan12_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
[4/118] cache FloorPlan13_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
[5/118] cache FloorPlan14_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
[6/118] cache FloorPlan15_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
[7/118] cache FloorPlan16_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
[8/118] cache FloorPlan17_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt
[9/118] cache FloorPlan18_step4_probe_

In [ ]:
required_metadata_keys = {
    "sample_id",
    "scene",
    "paragraph_id",
    "text",
    LABEL_FIELD,
    "evidence_type",
    "subject_uid",
    "object_uid",
    "_step6_sample_family",
    "_source_step5_pt_file",
    "_source_step5_input_dir",
    "_source_step5_absolute_pt_file",
    "_source_step5_drive_pt_file",
    "_source_local_record_idx",
}

missing_examples = []

for idx, meta in enumerate(metadata):
    missing = sorted(required_metadata_keys - set(meta.keys()))
    if missing:
        missing_examples.append({
            "record_index": idx,
            "sample_id": meta.get("sample_id"),
            "missing_keys": missing,
        })

if missing_examples:
    raise KeyError(
        "Some filtered metadata records are missing required keys. "
        f"First examples:\n{json.dumps(missing_examples[:10], indent=2, ensure_ascii=False)}"
    )

evidence_counts = summarize_counter(meta["evidence_type"] for meta in metadata)
label_counts = summarize_counter(meta[LABEL_FIELD] for meta in metadata)
scene_counts = summarize_counter(meta["scene"] for meta in metadata)

if set(evidence_counts.keys()) != {EXPLICIT_DIRECT}:
    raise ValueError(
        f"Current direct-only probe should retain only {EXPLICIT_DIRECT!r}, "
        f"but found evidence counts: {evidence_counts}"
    )

if DROP_NONE_LABEL and NONE_RELATION_LABEL in label_counts:
    raise ValueError(f"Found dropped label {NONE_RELATION_LABEL!r} in filtered metadata.")

if ALLOWED_LABELS is not None:
    disallowed = sorted(set(label_counts.keys()) - ALLOWED_LABELS)
    if disallowed:
        raise ValueError(f"Found labels outside STEP6_ALLOWED_LABELS: {disallowed}")

print("Filtered metadata validation passed.")
print("Evidence counts:", evidence_counts)
print("Label counts:", label_counts)
print("Number of scenes:", len(scene_counts))
print("First scene counts:")
for scene, count in list(scene_counts.items())[:20]:
    print(f"- {scene}: {count}")

Filtered metadata validation passed.
Evidence counts: {'explicit_direct': 21916}
Label counts: {'left_of': 4913, 'right_of': 4374, 'on': 4169, 'supports': 2901, 'above': 2392, 'below': 2243, 'in': 521, 'contains': 403}
Number of scenes: 118
First scene counts:
- FloorPlan30: 461
- FloorPlan13: 422
- FloorPlan17: 346
- FloorPlan5: 342
- FloorPlan20: 341
- FloorPlan1: 337
- FloorPlan22: 319
- FloorPlan7: 316
- FloorPlan16: 303
- FloorPlan29: 300
- FloorPlan28: 297
- FloorPlan6: 295
- FloorPlan14: 292
- FloorPlan4: 287
- FloorPlan24: 285
- FloorPlan10: 280
- FloorPlan19: 280
- FloorPlan9: 278
- FloorPlan27: 276
- FloorPlan26: 275


In [ ]:
# Streaming feature loader

def build_records_by_file(metadata: List[Dict[str, Any]]) -> Dict[str, List[Tuple[int, int]]]:
    records_by_file = defaultdict(list)

    for global_idx, meta in enumerate(metadata):
        pt_path = meta["_source_step5_absolute_pt_file"]
        local_record_idx = int(meta["_source_local_record_idx"])
        records_by_file[pt_path].append((global_idx, local_record_idx))

    return records_by_file


def load_features_for_one_layer(
    layer_idx: int,
    feature_key: str,
    metadata: List[Dict[str, Any]],
    records_by_file: Dict[str, List[Tuple[int, int]]],
) -> np.ndarray:
    if layer_idx < 0 or layer_idx >= NUM_LAYERS:
        raise ValueError(f"Invalid layer_idx={layer_idx}. Valid range: 0..{NUM_LAYERS - 1}")

    X_layer = np.empty((len(metadata), FEATURE_DIM), dtype=np.float32)

    for pt_path, index_pairs in records_by_file.items():
        payload = torch.load(pt_path, map_location="cpu")
        records = payload["records"]

        for global_idx, local_record_idx in index_pairs:
            rec = records[local_record_idx]

            if feature_key not in rec:
                raise KeyError(f"Missing feature key {feature_key!r} in {pt_path}")

            feature_arr = np.asarray(rec[feature_key])
            if feature_arr.shape != (NUM_LAYERS, FEATURE_DIM):
                raise ValueError(
                    f"Inconsistent feature shape in {pt_path}, record {local_record_idx}: "
                    f"expected {(NUM_LAYERS, FEATURE_DIM)}, got {feature_arr.shape}"
                )

            row = feature_arr[layer_idx].astype(np.float32)
            if row.shape != (FEATURE_DIM,):
                raise ValueError(
                    f"Invalid row shape at layer {layer_idx}: expected {(FEATURE_DIM,)}, got {row.shape}"
                )

            X_layer[global_idx] = row

        del payload
        del records
        gc.collect()

    return X_layer


RECORDS_BY_FILE = build_records_by_file(metadata)

print("Prepared streaming index.")
print("Number of active .pt files:", len(RECORDS_BY_FILE))
print("First source file:", next(iter(RECORDS_BY_FILE.keys())))

Prepared streaming index.
Number of active .pt files: 118
First source file: /content/step5_hs_cache_qwen2_5_3b_pair_ed_rel_ld_seen_unseen_dtp_only/pair/FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_3b.pt


In [ ]:
# Held-out bathroom scene split + seen/unseen directed type-pair test subsets

available_scenes = sorted({meta["scene"] for meta in metadata})
train_scene_set = set(TRAIN_SCENES)
test_scene_set = set(TEST_SCENES)

missing_train_scenes = sorted(train_scene_set - set(available_scenes))
missing_test_scenes = sorted(test_scene_set - set(available_scenes))

print("Available scenes after filtering:", len(available_scenes))

if missing_train_scenes:
    print("Warning: train scenes with no filtered records:", missing_train_scenes)

if missing_test_scenes:
    print("Warning: test scenes with no filtered records:", missing_test_scenes)

train_idx = np.asarray(
    [i for i, meta in enumerate(metadata) if meta["scene"] in train_scene_set],
    dtype=np.int64,
)

test_idx_full = np.asarray(
    [i for i, meta in enumerate(metadata) if meta["scene"] in test_scene_set],
    dtype=np.int64,
)

if len(train_idx) == 0:
    raise ValueError("No training examples after scene split.")

if len(test_idx_full) == 0:
    raise ValueError("No test examples after scene split.")


def get_directed_type_pair(meta: Dict[str, Any]) -> str:
    """
    Directed object-type pair used for the seen/unseen ablation.

    It follows the probe direction:
        subject_type -> object_type
    """
    subject_type = meta.get("subject_type")
    object_type = meta.get("object_type")

    if subject_type is None or object_type is None:
        raise KeyError(
            "Missing subject_type or object_type in metadata. "
            "This ablation requires Step5 metadata to preserve object types."
        )

    return f"{subject_type} -> {object_type}"


train_directed_type_pairs = {
    get_directed_type_pair(metadata[int(i)])
    for i in train_idx
}

test_directed_type_pairs = {
    get_directed_type_pair(metadata[int(i)])
    for i in test_idx_full
}

test_seen_directed_type_pairs = test_directed_type_pairs & train_directed_type_pairs
test_unseen_directed_type_pairs = test_directed_type_pairs - train_directed_type_pairs

test_seen_idx = np.asarray(
    [
        int(i)
        for i in test_idx_full
        if get_directed_type_pair(metadata[int(i)]) in train_directed_type_pairs
    ],
    dtype=np.int64,
)

test_unseen_idx = np.asarray(
    [
        int(i)
        for i in test_idx_full
        if get_directed_type_pair(metadata[int(i)]) not in train_directed_type_pairs
    ],
    dtype=np.int64,
)

available_test_subsets = {
    TEST_SUBSET_OVERALL_NAME: test_idx_full,
    TEST_SUBSET_SEEN_NAME: test_seen_idx,
    TEST_SUBSET_UNSEEN_NAME: test_unseen_idx,
}

if TEST_ABLATION_MODE == "none":
    test_subsets = {
        TEST_SUBSET_OVERALL_NAME: test_idx_full,
    }

elif TEST_ABLATION_MODE == "seen_unseen_train_directed_type_pair":
    test_subsets = {
        subset_name: available_test_subsets[subset_name]
        for subset_name in TEST_SUBSETS_TO_EVALUATE
    }

else:
    raise ValueError(f"Unsupported TEST_ABLATION_MODE: {TEST_ABLATION_MODE!r}")

for subset_name, subset_idx in test_subsets.items():
    if len(subset_idx) == 0:
        raise ValueError(f"Test subset {subset_name!r} is empty.")

if TEST_SUBSET_OVERALL_NAME in test_subsets:
    test_idx = test_subsets[TEST_SUBSET_OVERALL_NAME]
else:
    test_idx = next(iter(test_subsets.values()))

train_labels = set(y[train_idx].tolist())

for subset_name, subset_idx in test_subsets.items():
    subset_labels = set(y[subset_idx].tolist())
    missing_subset_labels = sorted(subset_labels - train_labels)

    if missing_subset_labels:
        raise ValueError(
            f"Some labels in test subset {subset_name!r} are absent from training labels. "
            f"Missing in train: {missing_subset_labels}"
        )


def summarize_indices(indices: np.ndarray) -> Dict[str, Any]:
    return {
        "num_examples": int(len(indices)),
        "label_counts": summarize_counter(y[indices].tolist()),
        "scene_counts": summarize_counter(metadata[int(i)]["scene"] for i in indices),
        "evidence_type_counts": summarize_counter(metadata[int(i)]["evidence_type"] for i in indices),
    }


train_summary = summarize_indices(train_idx)
test_summary = summarize_indices(test_idx)

test_subset_summaries = {
    subset_name: summarize_indices(subset_idx)
    for subset_name, subset_idx in test_subsets.items()
}

ablation_summary = {
    "test_ablation_mode": TEST_ABLATION_MODE,
    "test_pair_coverage_key": TEST_PAIR_COVERAGE_KEY,

    "train_num_examples": int(len(train_idx)),
    "full_test_num_examples": int(len(test_idx_full)),
    "seen_test_num_examples": int(len(test_seen_idx)),
    "unseen_test_num_examples": int(len(test_unseen_idx)),

    "num_train_directed_type_pairs": int(len(train_directed_type_pairs)),
    "num_full_test_directed_type_pairs": int(len(test_directed_type_pairs)),
    "num_test_directed_type_pairs_seen_in_train": int(len(test_seen_directed_type_pairs)),
    "num_test_directed_type_pairs_unseen_in_train": int(len(test_unseen_directed_type_pairs)),

    "directed_type_pair_coverage_ratio": (
        float(len(test_seen_directed_type_pairs) / len(test_directed_type_pairs))
        if test_directed_type_pairs else 0.0
    ),
    "sample_level_seen_directed_type_pair_ratio": (
        float(len(test_seen_idx) / len(test_idx_full))
        if len(test_idx_full) else 0.0
    ),
    "sample_level_unseen_directed_type_pair_ratio": (
        float(len(test_unseen_idx) / len(test_idx_full))
        if len(test_idx_full) else 0.0
    ),

    "test_subset_names": list(test_subsets.keys()),
}

print("Train summary:")
print(json.dumps(train_summary, indent=2, ensure_ascii=False))

print("\nTest subset summaries:")
print(json.dumps(test_subset_summaries, indent=2, ensure_ascii=False))

print("\nAblation summary:")
print(json.dumps(ablation_summary, indent=2, ensure_ascii=False))

Available scenes after filtering: 118
Train summary:
{
  "num_examples": 17226,
  "label_counts": {
    "left_of": 3789,
    "right_of": 3423,
    "on": 3301,
    "supports": 2252,
    "above": 1872,
    "below": 1821,
    "in": 434,
    "contains": 334
  },
  "scene_counts": {
    "FloorPlan30": 461,
    "FloorPlan13": 422,
    "FloorPlan17": 346,
    "FloorPlan5": 342,
    "FloorPlan20": 341,
    "FloorPlan1": 337,
    "FloorPlan22": 319,
    "FloorPlan7": 316,
    "FloorPlan16": 303,
    "FloorPlan29": 300,
    "FloorPlan28": 297,
    "FloorPlan6": 295,
    "FloorPlan14": 292,
    "FloorPlan4": 287,
    "FloorPlan24": 285,
    "FloorPlan10": 280,
    "FloorPlan19": 280,
    "FloorPlan9": 278,
    "FloorPlan27": 276,
    "FloorPlan26": 275,
    "FloorPlan325": 274,
    "FloorPlan11": 264,
    "FloorPlan8": 259,
    "FloorPlan25": 254,
    "FloorPlan326": 249,
    "FloorPlan227": 247,
    "FloorPlan203": 243,
    "FloorPlan2": 243,
    "FloorPlan218": 242,
    "FloorPlan23": 237,
    

In [ ]:
# Linear probe training and output row helpers

def train_probe_for_layer(X_train: np.ndarray, y_train: np.ndarray):
    if X_train.ndim != 2:
        raise ValueError(f"X_train must be 2D, got shape={X_train.shape}")
    if X_train.shape[0] != len(y_train):
        raise ValueError("X_train and y_train length mismatch.")
    if len(set(y_train.tolist())) < 2:
        raise ValueError("Need at least two training labels for logistic regression.")

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=MAX_ITER,
            C=C_VALUE,
            class_weight=CLASS_WEIGHT,
            solver=SOLVER,
            multi_class="auto",
            random_state=RANDOM_SEED,
        ),
    )

    clf.fit(X_train, y_train)
    return clf


def evaluate_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    label_order: List[str],
) -> Dict[str, Any]:
    report = classification_report(
        y_true,
        y_pred,
        labels=label_order,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(y_true, y_pred, labels=label_order)

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "classification_report": report,
        "confusion_matrix": cm,
        "num_examples": int(len(y_true)),
    }


def append_label_metrics_rows(rows, result, layer_idx: int, subset_name: str):
    report = result["classification_report"]

    for label in label_order:
        label_metrics = report.get(label, {})
        rows.append({
            "experiment_name": EXPERIMENT_NAME,
            "experiment_id": EXPERIMENT_ID,
            "script_family": SCRIPT_FAMILY,
            "subset": subset_name,
            "layer": int(layer_idx),
            "label": label,
            "precision": float(label_metrics.get("precision", 0.0)),
            "recall": float(label_metrics.get("recall", 0.0)),
            "f1_score": float(label_metrics.get("f1-score", 0.0)),
            "support": int(label_metrics.get("support", 0)),
            "accuracy": float(result["accuracy"]),
            "macro_f1": float(result["macro_f1"]),
            "num_examples": int(result["num_examples"]),
        })


def append_recall_matrix_rows(rows, result, layer_idx: int, subset_name: str):
    report = result["classification_report"]

    for label in label_order:
        label_metrics = report.get(label, {})
        rows.append({
            "experiment_name": EXPERIMENT_NAME,
            "experiment_id": EXPERIMENT_ID,
            "script_family": SCRIPT_FAMILY,
            "subset": subset_name,
            "layer": int(layer_idx),
            "label": label,
            "metric": "recall",
            "value": float(label_metrics.get("recall", 0.0)),
        })


def append_prediction_rows(
    rows,
    layer_idx: int,
    subset_name: str,
    indices: np.ndarray,
    y_true: np.ndarray,
    y_pred: np.ndarray,
):
    for local_i, global_idx in enumerate(indices):
        meta = metadata[int(global_idx)]

        directed_type_pair = None
        if meta.get("subject_type") is not None and meta.get("object_type") is not None:
            directed_type_pair = f"{meta.get('subject_type')} -> {meta.get('object_type')}"

        rows.append({
            "experiment_name": EXPERIMENT_NAME,
            "experiment_id": EXPERIMENT_ID,
            "script_family": SCRIPT_FAMILY,
            "subset": subset_name,
            "layer": int(layer_idx),

            "sample_id": meta.get("sample_id"),
            "pair_group_id": meta.get("pair_group_id"),
            "scene": meta.get("scene"),
            "paragraph_id": meta.get("paragraph_id"),

            "evidence_type": meta.get("evidence_type"),
            "probe_direction_relative_to_text": meta.get("probe_direction_relative_to_text"),
            "is_inverse_of_text_relation": meta.get("is_inverse_of_text_relation"),

            "true_label": str(y_true[local_i]),
            "pred_label": str(y_pred[local_i]),
            "is_correct": bool(y_true[local_i] == y_pred[local_i]),

            "subject_uid": meta.get("subject_uid"),
            "object_uid": meta.get("object_uid"),
            "subject_id": meta.get("subject_id"),
            "object_id": meta.get("object_id"),
            "subject_type": meta.get("subject_type"),
            "object_type": meta.get("object_type"),
            "directed_type_pair": directed_type_pair,

            "subject_alias": meta.get("subject_alias"),
            "object_alias": meta.get("object_alias"),

            "text_relation_label": meta.get("text_relation_label"),
            "text_subject_uid": meta.get("text_subject_uid"),
            "text_object_uid": meta.get("text_object_uid"),

            "evidence_sentence": meta.get("evidence_sentence"),
            "text": meta.get("text"),

            "_step6_sample_family": meta.get("_step6_sample_family"),
            "_source_step5_pt_file": meta.get("_source_step5_pt_file"),
            "_source_step5_input_dir": meta.get("_source_step5_input_dir"),
        })

In [ ]:
#  Layer-wise Step6 probe

layer_results = []
per_layer_label_metrics = []
recall_matrix_long_rows = []
prediction_rows = []

if LAYER_INDICES_TO_RUN is None:
    LAYER_INDICES_EVALUATED = list(range(NUM_LAYERS))
else:
    LAYER_INDICES_EVALUATED = [int(layer_idx) for layer_idx in LAYER_INDICES_TO_RUN]

invalid_layers = [
    layer_idx for layer_idx in LAYER_INDICES_EVALUATED
    if layer_idx < 0 or layer_idx >= NUM_LAYERS
]

if invalid_layers:
    raise ValueError(
        f"Invalid layer indices requested: {invalid_layers}. "
        f"Valid range: 0..{NUM_LAYERS - 1}."
    )

print("Layer indices to evaluate:", LAYER_INDICES_EVALUATED)
print("Test subsets:", {k: int(len(v)) for k, v in test_subsets.items()})

for layer_idx in LAYER_INDICES_EVALUATED:
    print("\n" + "=" * 100)
    print(f"Layer {layer_idx}/{NUM_LAYERS - 1}")
    print("=" * 100)

    X_layer = load_features_for_one_layer(
        layer_idx=layer_idx,
        feature_key=FEATURE_KEY,
        metadata=metadata,
        records_by_file=RECORDS_BY_FILE,
    )

    if X_layer.ndim != 2:
        raise ValueError(f"X_layer must be 2D, got shape={X_layer.shape}")
    if X_layer.shape[0] != len(metadata):
        raise ValueError("X_layer row count does not match metadata length.")
    if X_layer.shape[1] != FEATURE_DIM:
        raise ValueError("X_layer feature dimension does not match FEATURE_DIM.")

    print(
        f"Loaded X_layer shape={X_layer.shape}, memory_MB={X_layer.nbytes / 1024**2:.2f}",
        flush=True,
    )

    clf = train_probe_for_layer(
        X_train=X_layer[train_idx],
        y_train=y[train_idx],
    )

    train_pred = clf.predict(X_layer[train_idx])

    train_result = evaluate_predictions(
        y_true=y[train_idx],
        y_pred=train_pred,
        label_order=label_order,
    )

    layer_row = {
        "layer": int(layer_idx),
        "train": train_result,
    }

    for subset_name, subset_idx in test_subsets.items():
        subset_pred = clf.predict(X_layer[subset_idx])

        subset_result = evaluate_predictions(
            y_true=y[subset_idx],
            y_pred=subset_pred,
            label_order=label_order,
        )

        layer_row[subset_name] = subset_result

        append_label_metrics_rows(
            rows=per_layer_label_metrics,
            result=subset_result,
            layer_idx=layer_idx,
            subset_name=subset_name,
        )

        append_recall_matrix_rows(
            rows=recall_matrix_long_rows,
            result=subset_result,
            layer_idx=layer_idx,
            subset_name=subset_name,
        )

        append_prediction_rows(
            rows=prediction_rows,
            layer_idx=layer_idx,
            subset_name=subset_name,
            indices=subset_idx,
            y_true=y[subset_idx],
            y_pred=subset_pred,
        )

    layer_results.append(layer_row)

    metric_parts = []
    for subset_name in test_subsets.keys():
        subset_result = layer_row[subset_name]
        metric_parts.append(
            f"{subset_name}_macro_f1={subset_result['macro_f1']:.4f}, "
            f"{subset_name}_acc={subset_result['accuracy']:.4f}"
        )

    print(
        f"Layer {layer_idx} done | " + " | ".join(metric_parts),
        flush=True,
    )

    del X_layer
    del clf
    gc.collect()

print("\nLayer-wise probing complete.")
print("Number of layers evaluated:", len(layer_results))

Layer indices to evaluate: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36]
Test subsets: {'seen_train_directed_type_pair': 791, 'unseen_train_directed_type_pair': 3899}

Layer 0/36
Loaded X_layer shape=(21916, 2048), memory_MB=171.22


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer 0 done | seen_train_directed_type_pair_macro_f1=0.3766, seen_train_directed_type_pair_acc=0.5626 | unseen_train_directed_type_pair_macro_f1=0.1853, unseen_train_directed_type_pair_acc=0.2016

Layer 1/36
Loaded X_layer shape=(21916, 2048), memory_MB=171.22


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer 1 done | seen_train_directed_type_pair_macro_f1=0.7802, seen_train_directed_type_pair_acc=0.8369 | unseen_train_directed_type_pair_macro_f1=0.4471, unseen_train_directed_type_pair_acc=0.4676

Layer 2/36
Loaded X_layer shape=(21916, 2048), memory_MB=171.22


...

Layer 34 done | seen_train_directed_type_pair_macro_f1=0.5831, seen_train_directed_type_pair_acc=0.6966 | unseen_train_directed_type_pair_macro_f1=0.4989, unseen_train_directed_type_pair_acc=0.4958

Layer 35/36
Loaded X_layer shape=(21916, 2048), memory_MB=171.22


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer 35 done | seen_train_directed_type_pair_macro_f1=0.5650, seen_train_directed_type_pair_acc=0.6953 | unseen_train_directed_type_pair_macro_f1=0.4833, unseen_train_directed_type_pair_acc=0.4629

Layer 36/36
Loaded X_layer shape=(21916, 2048), memory_MB=171.22


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer 36 done | seen_train_directed_type_pair_macro_f1=0.5877, seen_train_directed_type_pair_acc=0.6966 | unseen_train_directed_type_pair_macro_f1=0.5228, unseen_train_directed_type_pair_acc=0.5012

Layer-wise probing complete.
Number of layers evaluated: 37


In [ ]:
# Save Step6 outputs using config-defined paths

if not layer_results:
    raise ValueError("layer_results is empty. Run Cell 8 first.")


def get_layer_metric(row, subset_key, metric_key):
    value = row.get(subset_key, {}).get(metric_key, None)
    return float("-inf") if value is None else float(value)


best_layer_summary = {}

for subset_name in test_subsets.keys():
    best_layer = max(
        layer_results,
        key=lambda row: get_layer_metric(row, subset_name, "macro_f1"),
    )

    best_layer_summary[f"best_{subset_name}_macro_f1"] = {
        "layer": int(best_layer["layer"]),
        "macro_f1": float(best_layer[subset_name]["macro_f1"]),
        "accuracy": float(best_layer[subset_name]["accuracy"]),
        "num_examples": int(best_layer[subset_name]["num_examples"]),
    }

layer_score_rows = []

for row in layer_results:
    base_row = {
        "experiment_name": EXPERIMENT_NAME,
        "experiment_id": EXPERIMENT_ID,
        "script_family": SCRIPT_FAMILY,
        "layer": int(row["layer"]),

        "train_accuracy": float(row["train"]["accuracy"]),
        "train_macro_f1": float(row["train"]["macro_f1"]),
    }

    for subset_name in test_subsets.keys():
        subset_result = row[subset_name]
        prefix = subset_name

        base_row[f"{prefix}_test_accuracy"] = float(subset_result["accuracy"])
        base_row[f"{prefix}_test_macro_f1"] = float(subset_result["macro_f1"])
        base_row[f"{prefix}_num_examples"] = int(subset_result["num_examples"])

    layer_score_rows.append(base_row)

layer_scores_df = pd.DataFrame(layer_score_rows)
metrics_df = pd.DataFrame(per_layer_label_metrics)
recall_matrix_long_df = pd.DataFrame(recall_matrix_long_rows)
predictions_df = pd.DataFrame(prediction_rows)

merged_record_summary = {
    "num_records_before_filtering": int(raw_record_count_total),
    "num_records_after_filtering": int(len(metadata)),
    "num_records_dropped_by_step6_filter": int(raw_record_count_total - len(metadata)),
    "sample_family_counts": summarize_counter(meta.get("_step6_sample_family") for meta in metadata),
    "evidence_type_counts": summarize_counter(meta.get("evidence_type") for meta in metadata),
    "label_counts": summarize_counter(y.tolist()),
    "metadata_keys": sorted(set().union(*(m.keys() for m in metadata))),
    "feature_loading_mode": "streaming_by_layer",
    "use_local_content_cache": bool(USE_LOCAL_CONTENT_CACHE),
    "local_content_cache_root": LOCAL_CONTENT_CACHE_ROOT if USE_LOCAL_CONTENT_CACHE else None,
}

step6_config_summary = {
    "experiment_id": EXPERIMENT_ID,
    "experiment_name": EXPERIMENT_NAME,
    "experiment_short_name": EXPERIMENT_SHORT_NAME,
    "experiment_description": EXPERIMENT_DESCRIPTION,
    "script_family": SCRIPT_FAMILY,

    "model_name": MODEL_NAME,
    "model_tag": MODEL_TAG,

    "sample_family": SAMPLE_FAMILY,
    "feature_key": FEATURE_KEY,
    "label_field": LABEL_FIELD,

    "keep_evidence_types": sorted(KEEP_EVIDENCE_TYPES),
    "drop_none_label": bool(DROP_NONE_LABEL),
    "drop_empty_label": bool(DROP_EMPTY_LABEL),
    "none_relation_label": NONE_RELATION_LABEL,
    "allowed_labels": None if ALLOWED_LABELS is None else sorted(ALLOWED_LABELS),

    "test_ablation_mode": TEST_ABLATION_MODE,
    "test_pair_coverage_key": TEST_PAIR_COVERAGE_KEY,
    "test_include_full_bathroom": TEST_INCLUDE_FULL_BATHROOM,
    "test_subset_names": list(test_subsets.keys()),

    "logreg_max_iter": int(MAX_ITER),
    "logreg_C": float(C_VALUE),
    "logreg_class_weight": CLASS_WEIGHT,
    "logreg_solver": SOLVER,

    "random_seed": int(RANDOM_SEED),
    "step6_input_dir": STEP6_INPUT_DIR,
    "step6_output_dir": STEP6_OUTPUT_DIR,

    "train_scenes": TRAIN_SCENES,
    "test_scenes": TEST_SCENES,
    "require_explicit_scene_split": bool(REQUIRE_EXPLICIT_SCENE_SPLIT),

    "num_layers": int(NUM_LAYERS),
    "feature_dim": int(FEATURE_DIM),
    "layer_indices_evaluated": [int(x) for x in LAYER_INDICES_EVALUATED],
}

full_results = {
    "experiment": step6_config_summary,
    "record_summary": merged_record_summary,
    "source_file_summaries": source_file_summaries,
    "split_summary": {
        "train": train_summary,
        "default_test": test_summary,
        "test_subsets": test_subset_summaries,
        "ablation": ablation_summary,
    },
    "label_order": label_order,
    "best_layer_summary": best_layer_summary,
    "layer_results": make_json_safe(layer_results),
}

manifest = {
    "experiment_name": EXPERIMENT_NAME,
    "experiment_id": EXPERIMENT_ID,
    "script_family": SCRIPT_FAMILY,
    "model_name": MODEL_NAME,
    "model_tag": MODEL_TAG,
    "feature_key": FEATURE_KEY,
    "label_field": LABEL_FIELD,
    "sample_family": SAMPLE_FAMILY,
    "keep_evidence_types": sorted(KEEP_EVIDENCE_TYPES),
    "allowed_labels": None if ALLOWED_LABELS is None else sorted(ALLOWED_LABELS),

    "test_ablation_mode": TEST_ABLATION_MODE,
    "test_pair_coverage_key": TEST_PAIR_COVERAGE_KEY,
    "test_subset_names": list(test_subsets.keys()),
    "ablation_summary": ablation_summary,

    "train_num_examples": int(len(train_idx)),
    "default_test_num_examples": int(len(test_idx)),
    "test_subset_num_examples": {
        subset_name: int(len(subset_idx))
        for subset_name, subset_idx in test_subsets.items()
    },

    "num_layers": int(NUM_LAYERS),
    "feature_dim": int(FEATURE_DIM),
    "layer_indices_evaluated": [int(x) for x in LAYER_INDICES_EVALUATED],
    "best_layer_summary": best_layer_summary,
    "output_files": {
        "full_result_json": STEP6_RESULT_JSON,
        "layer_scores_csv": STEP6_LAYER_SCORES_CSV,
        "per_layer_label_metrics_csv": STEP6_LABEL_METRICS_CSV,
        "recall_matrix_long_csv": STEP6_RECALL_MATRIX_LONG_CSV,
        "test_predictions_by_layer_csv": STEP6_TEST_PREDICTIONS_CSV,
        "manifest_json": STEP6_MANIFEST_JSON,
    },
}

os.makedirs(STEP6_OUTPUT_DIR, exist_ok=True)

with open(STEP6_RESULT_JSON, "w", encoding="utf-8") as f:
    json.dump(make_json_safe(full_results), f, indent=2, ensure_ascii=False)

layer_scores_df.to_csv(STEP6_LAYER_SCORES_CSV, index=False)
metrics_df.to_csv(STEP6_LABEL_METRICS_CSV, index=False)
recall_matrix_long_df.to_csv(STEP6_RECALL_MATRIX_LONG_CSV, index=False)
predictions_df.to_csv(STEP6_TEST_PREDICTIONS_CSV, index=False)

with open(STEP6_MANIFEST_JSON, "w", encoding="utf-8") as f:
    json.dump(make_json_safe(manifest), f, indent=2, ensure_ascii=False)

print("Saved Step6 outputs to:")
print("STEP6_OUTPUT_DIR:", STEP6_OUTPUT_DIR)
print("Full result JSON:", STEP6_RESULT_JSON)
print("Layer scores CSV:", STEP6_LAYER_SCORES_CSV)
print("Per-layer label metrics CSV:", STEP6_LABEL_METRICS_CSV)
print("Recall matrix long CSV:", STEP6_RECALL_MATRIX_LONG_CSV)
print("Test predictions CSV:", STEP6_TEST_PREDICTIONS_CSV)
print("Manifest JSON:", STEP6_MANIFEST_JSON)

print("\nBest layer summary:")
print(json.dumps(make_json_safe(best_layer_summary), indent=2, ensure_ascii=False))

Saved Step6 outputs to:
STEP6_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only
Full result JSON: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only_full_results.json
Layer scores CSV: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only_layer_scores.c

In [ ]:
# Result integrity check

print("=" * 100)
print("Step6 result integrity check")
print("=" * 100)

expected_output_files = {
    "full_result_json": STEP6_RESULT_JSON,
    "layer_scores_csv": STEP6_LAYER_SCORES_CSV,
    "per_layer_label_metrics_csv": STEP6_LABEL_METRICS_CSV,
    "recall_matrix_long_csv": STEP6_RECALL_MATRIX_LONG_CSV,
    "test_predictions_by_layer_csv": STEP6_TEST_PREDICTIONS_CSV,
    "manifest_json": STEP6_MANIFEST_JSON,
}

missing_files = {
    name: path for name, path in expected_output_files.items()
    if not os.path.exists(path)
}

if missing_files:
    raise FileNotFoundError(
        "Some expected Step6 output files are missing:\n"
        + json.dumps(missing_files, indent=2, ensure_ascii=False)
    )

layer_scores_check_df = pd.read_csv(STEP6_LAYER_SCORES_CSV)
metrics_check_df = pd.read_csv(STEP6_LABEL_METRICS_CSV)
recall_check_df = pd.read_csv(STEP6_RECALL_MATRIX_LONG_CSV)
predictions_check_df = pd.read_csv(STEP6_TEST_PREDICTIONS_CSV)

required_layer_score_cols = {
    "experiment_name",
    "experiment_id",
    "script_family",
    "layer",
    "train_accuracy",
    "train_macro_f1",
}

# Each configured test subset should have its own columns in layer_scores.
for subset_name in test_subsets.keys():
    required_layer_score_cols.update({
        f"{subset_name}_test_accuracy",
        f"{subset_name}_test_macro_f1",
        f"{subset_name}_num_examples",
    })

required_metrics_cols = {
    "experiment_name",
    "experiment_id",
    "script_family",
    "subset",
    "layer",
    "label",
    "precision",
    "recall",
    "f1_score",
    "support",
    "accuracy",
    "macro_f1",
    "num_examples",
}

required_recall_cols = {
    "experiment_name",
    "experiment_id",
    "script_family",
    "subset",
    "layer",
    "label",
    "metric",
    "value",
}

required_prediction_cols = {
    "experiment_name",
    "experiment_id",
    "script_family",
    "subset",
    "layer",
    "sample_id",
    "scene",
    "paragraph_id",
    "evidence_type",
    "true_label",
    "pred_label",
    "is_correct",
    "subject_uid",
    "object_uid",
    "subject_type",
    "object_type",
    "directed_type_pair",
    "text",
    "_step6_sample_family",
    "_source_step5_pt_file",
    "_source_step5_input_dir",
}


def check_required_columns(df, required_cols, label):
    missing = sorted(required_cols - set(df.columns))
    if missing:
        raise ValueError(f"{label} missing required columns: {missing}")
    return {"rows": int(len(df)), "columns": int(len(df.columns))}


column_check_summary = {
    "layer_scores": check_required_columns(
        layer_scores_check_df,
        required_layer_score_cols,
        "layer_scores",
    ),
    "per_layer_label_metrics": check_required_columns(
        metrics_check_df,
        required_metrics_cols,
        "per_layer_label_metrics",
    ),
    "recall_matrix_long": check_required_columns(
        recall_check_df,
        required_recall_cols,
        "recall_matrix_long",
    ),
    "test_predictions_by_layer": check_required_columns(
        predictions_check_df,
        required_prediction_cols,
        "test_predictions_by_layer",
    ),
}

# ------------------------------------------------------------
# Layer coverage check
# ------------------------------------------------------------

expected_layers = set(int(x) for x in LAYER_INDICES_EVALUATED)
observed_layers = set(int(x) for x in layer_scores_check_df["layer"].unique())

if observed_layers != expected_layers:
    raise ValueError(
        f"Layer coverage mismatch. expected={sorted(expected_layers)}, "
        f"observed={sorted(observed_layers)}"
    )

# Subset coverage check

expected_subsets = set(test_subsets.keys())

observed_metric_subsets = set(metrics_check_df["subset"].astype(str).unique())
observed_recall_subsets = set(recall_check_df["subset"].astype(str).unique())
observed_prediction_subsets = set(predictions_check_df["subset"].astype(str).unique())

if observed_metric_subsets != expected_subsets:
    raise ValueError(
        f"Metric subset mismatch. expected={sorted(expected_subsets)}, "
        f"observed={sorted(observed_metric_subsets)}"
    )

if observed_recall_subsets != expected_subsets:
    raise ValueError(
        f"Recall subset mismatch. expected={sorted(expected_subsets)}, "
        f"observed={sorted(observed_recall_subsets)}"
    )

if observed_prediction_subsets != expected_subsets:
    raise ValueError(
        f"Prediction subset mismatch. expected={sorted(expected_subsets)}, "
        f"observed={sorted(observed_prediction_subsets)}"
    )

# Check prediction row counts.
# Rows are expected for every enabled subset at every layer.

expected_prediction_rows = (
    sum(int(len(subset_idx)) for subset_idx in test_subsets.values())
    * len(LAYER_INDICES_EVALUATED)
)

if len(predictions_check_df) != expected_prediction_rows:
    raise ValueError(
        f"Prediction row count mismatch: expected={expected_prediction_rows}, "
        f"observed={len(predictions_check_df)}"
    )

# Per-subset prediction row count check.
prediction_row_count_by_subset = (
    predictions_check_df.groupby("subset")
    .size()
    .to_dict()
)

expected_prediction_row_count_by_subset = {
    subset_name: int(len(subset_idx)) * len(LAYER_INDICES_EVALUATED)
    for subset_name, subset_idx in test_subsets.items()
}

if prediction_row_count_by_subset != expected_prediction_row_count_by_subset:
    raise ValueError(
        "Per-subset prediction row count mismatch.\n"
        f"expected={expected_prediction_row_count_by_subset}\n"
        f"observed={prediction_row_count_by_subset}"
    )

# Content checks

if set(predictions_check_df["evidence_type"].astype(str).unique()) != {EXPLICIT_DIRECT}:
    raise ValueError("Predictions contain non-explicit_direct evidence types.")

if set(predictions_check_df["true_label"].astype(str).unique()) - set(label_order):
    raise ValueError("Predictions contain labels outside label_order.")

with open(STEP6_MANIFEST_JSON, "r", encoding="utf-8") as f:
    saved_manifest = json.load(f)

if saved_manifest["experiment_name"] != EXPERIMENT_NAME:
    raise ValueError("Manifest experiment_name mismatch.")

if set(saved_manifest.get("test_subset_names", [])) != expected_subsets:
    raise ValueError(
        "Manifest test_subset_names mismatch.\n"
        f"expected={sorted(expected_subsets)}\n"
        f"manifest={saved_manifest.get('test_subset_names')}"
    )

integrity_summary = {
    "status": "passed",
    "experiment_name": EXPERIMENT_NAME,
    "output_dir": STEP6_OUTPUT_DIR,
    "column_check_summary": column_check_summary,
    "num_layers_checked": int(len(expected_layers)),
    "expected_subsets": sorted(expected_subsets),
    "num_prediction_rows": int(len(predictions_check_df)),
    "prediction_row_count_by_subset": make_json_safe(prediction_row_count_by_subset),
    "expected_prediction_row_count_by_subset": make_json_safe(expected_prediction_row_count_by_subset),
    "best_layer_summary": saved_manifest.get("best_layer_summary"),
}

integrity_summary_path = os.path.join(
    STEP6_OUTPUT_DIR,
    f"{EXPERIMENT_NAME}_integrity_summary.json",
)

with open(integrity_summary_path, "w", encoding="utf-8") as f:
    json.dump(make_json_safe(integrity_summary), f, indent=2, ensure_ascii=False)

print("Integrity check passed.")
print(json.dumps(make_json_safe(integrity_summary), indent=2, ensure_ascii=False))
print("Integrity summary:", integrity_summary_path)

Step6 result integrity check
Integrity check passed.
{
  "status": "passed",
  "experiment_name": "step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only",
  "output_dir": "/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only",
  "column_check_summary": {
    "layer_scores": {
      "rows": 37,
      "columns": 12
    },
    "per_layer_label_metrics": {
      "rows": 592,
      "columns": 13
    },
    "recall_matrix_long": {
      "rows": 592,
      "columns": 8
    },
    "test_predictions_by_layer": {
      "rows": 173530,
      "columns": 32
    }
  },
  "num_layers_checked": 37,
  "expected_subsets": [
    "seen_train_directed_type_pair",
    "unseen_train_directed_type_pair"
  ],
  "num_prediction_rows": 173530,
  "prediction_row_count_by_subset": {
    "

In [ ]:
from google.colab import files

output_files = sorted(os.listdir(STEP6_OUTPUT_DIR))

if not output_files:
    raise FileNotFoundError(f"No files found in STEP6_OUTPUT_DIR: {STEP6_OUTPUT_DIR}")

zip_base = f"/content/{EXPERIMENT_NAME}"
zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=STEP6_OUTPUT_DIR,
)

print("Created zip:", zip_path)
print("STEP6_OUTPUT_DIR:", STEP6_OUTPUT_DIR)
print("Number of files included:", len(output_files))

print("\nFiles included:")
for name in output_files:
    print("-", name)

files.download(zip_path)

Created zip: /content/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only.zip
STEP6_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only
Number of files included: 7

Files included:
- step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only_full_results.json
- step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only_integrity_summary.json
- step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only_layer_scores.csv
- step6_pilot_full_ithor120_diverse_qwen2_5_3b_pair_explicit_direct_relation_ld_heldout_bathroom_test_seen_unseen_dtp_only_manifest.json
- step6_pi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>